# Unidad 5 — Sistemas Multi-Agente Modernos
## Notebook 01: Fundamentos de Agentes Modernos `[Principiante ★★☆☆☆]`

**Duración estimada:** 4-5 horas  
**Entorno:** `ia_multiagente` (Python 3.11)  
**Modelo por defecto:** Gemini 2.0 Flash (gratuito) · Alternativa: Ollama Llama 3.2  
**Costo estimado:** ~$0.01 con Gemini Flash · $0 con Ollama  
**Skill que se crea:** `external_skills/agent_warmup/context_loader.py`

---

## Tabla de Contenidos

1. [Preparación del Entorno (Warm-Up)](#1-preparacion)
2. [¿Qué es un Agente de IA?](#2-que-es-agente)
   - 2.1 El Ciclo Percibir-Razonar-Actuar
   - 2.2 Taxonomía de Agentes
   - 2.3 Agentes vs. Cadenas vs. Modelos
3. [ReAct: el Paradigma Dominante](#3-react)
   - 3.1 Razonamiento + Actuación
   - 3.2 Function Calling vs Tool Use
   - 3.3 Implementación con LangChain
4. [Skills y Warm-Ups](#4-skills)
   - 4.1 ¿Qué es una Skill?
   - 4.2 El patrón Warm-Up
   - 4.3 Skill Registry
   - 4.4 Construir `context_loader.py` paso a paso
5. [LangChain LCEL: Composición de Cadenas](#5-lcel)
   - 5.1 El operador `|`
   - 5.2 Agente con tools usando LCEL
6. [Los 4 Tipos de Memoria en Agentes](#6-memoria)
7. [Patrones de Razonamiento Avanzado](#7-patrones)
   - 7.1 Chain of Thought
   - 7.2 Self-Reflection
8. [Proyecto: Asistente de Investigación con Warm-Up](#8-proyecto)
9. [Resumen y Criterios de Evaluación](#9-resumen)

---

## Objetivos de Aprendizaje

Al completar esta notebook serás capaz de:
- Explicar el ciclo Percibir-Razonar-Actuar y ubicar cada componente en código real
- Distinguir entre tool, skill y agente — tres conceptos que los principiantes suelen confundir
- Implementar un agente ReAct con al menos 2 tools usando LangChain LCEL
- Describir los 4 tipos de memoria con ejemplos concretos
- Crear y registrar una skill propia (`context_loader`) en el Skill Registry del proyecto

**Prerequisito mínimo:** Python intermedio (listas, dicts, funciones, clases básicas)

---
## 1. Preparación del Entorno (Warm-Up) <a id='1-preparacion'></a>

Antes de escribir una línea de código de agentes, verificamos que el entorno está listo. Este patrón de celda de warm-up es **obligatorio** en todas las notebooks de la unidad.


### Configuracion de Proveedores de LLM

Esta notebook soporta **cuatro backends**. La logica de seleccion es automatica segun las keys disponibles en `.env`:

| Prioridad | Backend | Variable en `.env` | Costo |
|---|---|---|---|
| 1 | **OpenRouter** | `OPENROUTER_API_KEY=sk-or-...` | Gratis con modelos `:free` |
| 2 | Google Gemini | `GOOGLE_API_KEY=...` | Gratis hasta 1M tokens/dia |
| 3 | OpenAI | `OPENAI_API_KEY=sk-...` | De pago |
| 4 | Ollama local | *(sin key)* | Gratis, requiere hardware local |

**Setup de OpenRouter (recomendado):**

1. Registrate en [openrouter.ai](https://openrouter.ai) → sección **Keys**
2. Genera y copia la clave `sk-or-...`
3. Agrega al `.env` en la raiz del proyecto:

```
OPENROUTER_API_KEY=sk-or-xxxxxxxxxxxxxxxx
OPENROUTER_MODEL=meta-llama/llama-3.2-3b-instruct:free
```

**Modelos gratuitos recomendados para este curso:**

| Modelo | ID para `OPENROUTER_MODEL` |
|---|---|
| Llama 3.2 3B | `meta-llama/llama-3.2-3b-instruct:free` |
| Mistral 7B | `mistralai/mistral-7b-instruct:free` |
| Gemma 3 12B | `google/gemma-3-12b-it:free` |
| Llama 3.1 8B Meta | `meta-llama/llama-3.1-8b-instruct:free` |

Lista completa con modelos gratuitos: https://openrouter.ai/models?q=free


In [1]:

# ============================================================
# WARM-UP: Entorno y Dependencias — U5_01
# ============================================================
# Paso 1: Verificar instalaciones necesarias para esta notebook
import subprocess, sys

def check_install(package, import_name=None):
    """Verifica si un paquete está instalado; lo instala si no."""
    name = import_name or package.split('==')[0].replace('-', '_')
    try:
        __import__(name)
        print(f"  OK: {package}")
    except ImportError:
        print(f"  Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"  Instalado: {package}")

packages = [
    ("langchain==0.3.25",           "langchain"),
    ("langchain-community==0.3.24",  "langchain_community"),
    ("langchain-core==0.3.55",       "langchain_core"),
    ("langchain-openai==0.3.12",     "langchain_openai"),      # OpenRouter usa protocolo OpenAI
    ("langchain-google-genai==2.1.4","langchain_google_genai"),
    # ("langchain-ollama==0.3.2",      "langchain_ollama"),
    ("python-dotenv==1.1.0",         "dotenv"),
]

print("Verificando paquetes...")
for pkg, imp in packages:
    check_install(pkg, imp)

# Paso 2: Cargar variables de entorno desde .env
from dotenv import load_dotenv
import os

# Busca .env en la carpeta actual y directorios padre
load_dotenv(override=True)

# Paso 3: Verificar qué LLMs están configurados
google_key      = os.environ.get("GOOGLE_API_KEY")
# openai_key      = os.environ.get("OPENAI_API_KEY")
openrouter_key  = os.environ.get("OPENROUTER_API_KEY")
# ollama_url      = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")

print("\n--- Backends disponibles ---")
if openrouter_key:
    print("  [*] OpenRouter      — OPENROUTER_API_KEY encontrada (prioridad 1)")
if google_key:
    print("  [*] Google Gemini   — GOOGLE_API_KEY encontrada")
# if openai_key:
    # print("  [*] OpenAI          — OPENAI_API_KEY encontrada")
# print(f"  [ ] Ollama local    — {ollama_url}  (sin API key requerida)")

if not any([openrouter_key, google_key]):
    print(
        "\nATENCION: No se encontró ninguna API key en .env.\n"
        "Opciones recomendadas:\n"
        "  1. OpenRouter (accede a 200+ modelos con una sola key):\n"
        "     https://openrouter.ai  ->  agrega OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Google Gemini Flash (gratis hasta 1M tokens/día):\n"
        "     https://aistudio.google.com/app/apikey  ->  GOOGLE_API_KEY=...\n"
        # f"  3. Ollama local: ollama pull llama3.2  (servidor en {ollama_url})"
    )

print("\nWarm-up completado.")


Verificando paquetes...
  OK: langchain==0.3.25
  OK: langchain-community==0.3.24
  OK: langchain-core==0.3.55
  OK: langchain-openai==0.3.12
  OK: langchain-google-genai==2.1.4
  OK: python-dotenv==1.1.0

--- Backends disponibles ---
  [*] OpenRouter      — OPENROUTER_API_KEY encontrada (prioridad 1)
  [*] Google Gemini   — GOOGLE_API_KEY encontrada

Warm-up completado.


In [2]:
# Paso 4: Configurar el LLM por defecto para toda la notebook
# ----------------------------------------------------------------
# PRIORIDAD:
#   1. OpenRouter  — una key para 200+ modelos (Llama, Claude, Mistral, Qwen...)
#      Si el modelo preferido esta saturado (429), prueba el siguiente de la lista
#   2. Gemini Flash — gratis, alta velocidad
#   3. OpenAI       — GPT-4o-mini  (comentado)
#   4. Ollama local — sin costo, sin internet  (comentado)
# ----------------------------------------------------------------
# Para cambiar el modelo preferido edita OPENROUTER_MODEL en tu .env
# Lista completa de modelos: https://openrouter.ai/models?q=free
# ----------------------------------------------------------------

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from openai import RateLimitError as OpenAIRateLimitError
# from langchain_ollama import ChatOllama

# Recargar .env — aplica cambios sin reiniciar el kernel
load_dotenv(override=True)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Lista de modelos gratuitos en orden de preferencia.
# El modelo de .env va primero; si da 429, se prueban los siguientes automaticamente.
OPENROUTER_FREE_MODELS = [
    os.environ.get("OPENROUTER_MODEL", "google/gemma-3-12b-it:free"),
    "google/gemma-3-12b-it:free",
    "mistralai/mistral-7b-instruct:free",
    "qwen/qwen-2.5-72b-instruct:free",
    "meta-llama/llama-3.2-3b-instruct:free",
]
# Eliminar duplicados manteniendo orden
_seen = set()
OPENROUTER_FREE_MODELS = [m for m in OPENROUTER_FREE_MODELS if not (m in _seen or _seen.add(m))]

llm = None
OPENROUTER_MODEL = None

if os.environ.get("OPENROUTER_API_KEY"):
    for _model_id in OPENROUTER_FREE_MODELS:
        try:
            _candidate = ChatOpenAI(
                base_url=OPENROUTER_BASE_URL,
                api_key=os.environ["OPENROUTER_API_KEY"],
                model=_model_id,
                temperature=0,
                default_headers={
                    "HTTP-Referer": "https://github.com/antigravity-nano",
                    "X-Title": "Antigravity Nano IA Course",
                },
            )
            _test = _candidate.invoke("Responde solo con 'OK': estas funcionando?")
            llm = _candidate
            OPENROUTER_MODEL = _model_id
            print(f"Usando: OpenRouter — {_model_id}")
            print("  Cambia el modelo preferido en .env con OPENROUTER_MODEL=<id>")
            print("  Modelos gratis: https://openrouter.ai/models?q=free")
            print(f"\nPrueba LLM: {_test.content}")
            break
        except OpenAIRateLimitError:
            print(f"  [!] {_model_id} saturado (429) — probando siguiente...")
        except Exception as _e:
            print(f"  [!] {_model_id} error: {_e} — probando siguiente...")

if llm is None and os.environ.get("GOOGLE_API_KEY"):
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0,
        google_api_key=os.environ["GOOGLE_API_KEY"],
    )
    OPENROUTER_MODEL = None
    print("Usando: Gemini 2.0 Flash (Google AI)")
    _test = llm.invoke("Responde solo con 'OK': estas funcionando?")
    print(f"\nPrueba LLM: {_test.content}")

# elif os.environ.get("OPENAI_API_KEY"):
    # llm = ChatOpenAI(
        # model="gpt-4o-mini",
        # temperature=0,
        # api_key=os.environ["OPENAI_API_KEY"],
    # )
    # print("Usando: GPT-4o-mini (OpenAI)")

# else:
    # llm = ChatOllama(model="llama3.2", temperature=0)
    # print("Usando: Llama 3.2 via Ollama (local)")

if llm is None:
    raise RuntimeError(
        "No se encontro ningun LLM disponible.\n"
        "Verifica que OPENROUTER_API_KEY o GOOGLE_API_KEY esten en tu .env"
    )


Usando: OpenRouter — google/gemini-2.5-flash
  Cambia el modelo preferido en .env con OPENROUTER_MODEL=<id>
  Modelos gratis: https://openrouter.ai/models?q=free

Prueba LLM: OK


---
## 2. ¿Qué es un Agente de IA? <a id='2-que-es-agente'></a>

### 2.1 El Ciclo Percibir-Razonar-Actuar

Antes de escribir una línea de código, necesitamos entender exactamente **qué problema resuelven los agentes** y por qué surgieron como paradigma dominante en 2024-2026.

Un modelo de lenguaje por sí solo es una función de texto a texto: recibe un prompt, genera una respuesta. Eso es poderoso, pero limitado. No puede ejecutar código, no puede consultar una base de datos en tiempo real, no puede tomar acciones en el mundo. Un **agente** rompe esa limitación.

Formalmente, un agente de IA es un sistema que opera en un ciclo continuo:

$$
\text{Agente}: S \times O \rightarrow A
$$

donde $S$ es el estado interno (memoria, contexto), $O$ es la observación del entorno (input del usuario, resultado de tools), y $A$ es la acción a tomar (invocar una tool, generar texto, pedir más información).

El ciclo completo se puede expresar como:

$$
o_{t+1} = \text{env}(o_t, a_t) \quad \text{donde} \quad a_t = \pi(o_t, s_t)
$$

$\pi$ es la *política* del agente (el LLM + sus instrucciones), que decide qué acción tomar dado el estado actual de la observación.

```
┌─────────────────────────────────────────────────────┐
│               CICLO DEL AGENTE                      │
│                                                     │
│   Input / Observación                               │
│         │                                           │
│         ▼                                           │
│   ┌─────────────┐                                   │
│   │  PERCIBIR   │  ← Contexto, historial, memoria  │
│   └──────┬──────┘                                   │
│          │                                          │
│          ▼                                          │
│   ┌─────────────┐                                   │
│   │  RAZONAR    │  ← LLM (política π)              │
│   └──────┬──────┘                                   │
│          │                                          │
│          ▼                                          │
│   ┌─────────────┐                                   │
│   │   ACTUAR    │  → Tool call, respuesta, stop    │
│   └──────┬──────┘                                   │
│          │                                          │
│          └──────────────────► observación t+1       │
│                              (el ciclo continúa)    │
└─────────────────────────────────────────────────────┘
```

### 2.2 Taxonomía de Agentes

No todos los agentes son iguales. La literatura de IA distingue tres familias principales:

| Tipo | Modelo cognitivo | Ejemplo en LLMs | Cuándo usarlo |
|------|-----------------|-----------------|---------------|
| **Reflexivo** | Estímulo → Respuesta directa, sin planificación | Un LLM puro sin tools | Tareas simples de Q&A |
| **Deliberativo** | Planifica antes de actuar, construye un plan de pasos | ReAct, CoT | Tareas complejas multi-paso |
| **BDI** (Belief-Desire-Intention) | Mantiene modelo del mundo (creencias), objetivos (deseos) y compromisos (intenciones) | AutoGen / AG2, agentes con memoria episódica | Sistemas autónomos de larga duración |

En el ecosistema actual (2026), la mayoría de los agentes productivos son **deliberativos** implementando el patrón **ReAct** (que veremos en la sección 3).

### 2.3 Agentes vs. Cadenas vs. Modelos

Una confusión frecuente entre principiantes es mezclar estos tres conceptos:

| Concepto | Descripción | ¿Puede usar tools? | ¿Tiene ciclos? | Ejemplo |
|----------|-------------|-------------------|----------------|----------|
| **Modelo** | LLM puro, genera texto | No | No | `llm.invoke("...")`|
| **Cadena** | Secuencia fija de pasos | Solo via steps predefinidos | No | `prompt \| llm \| parser` |
| **Agente** | El LLM decide dinámicamente qué hacer a continuación | Sí, en tiempo de inferencia | Sí | `AgentExecutor` con tools |

La diferencia clave: en una **cadena**, tú (el programador) defines el flujo. En un **agente**, el LLM decide el flujo en tiempo de ejecución.

In [3]:
# ────────────────────────────────────────────────────────────
# Output esperado: diferencia entre modelo, cadena y agente
# ────────────────────────────────────────────────────────────
# Paso 1: Modelo puro — una sola llamada, sin tools
from langchain_core.messages import HumanMessage

response_model = llm.invoke([HumanMessage(content="¿Cuánto es 2 + 2? Solo el número.")])
print("Modelo puro:", response_model.content)

# Paso 2: Cadena simple con LCEL — flujo fijo definido por el programador
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("Traduce al inglés: {texto}")
    | llm
    | StrOutputParser()
)
response_chain = chain.invoke({"texto": "Hola, ¿cómo estás?"})
print("Cadena:", response_chain)

# Paso 3: El agente lo veremos en la sección 3 — requiere tools

Modelo puro: 4
Cadena: "Hola, ¿cómo estás?" se traduce al inglés como:

**"Hello, how are you?"**


**Interpretación:** El modelo devuelve texto directamente. La cadena aplica una transformación fija (prompt → LLM → parseo). Ninguno de los dos puede, por ejemplo, ir a buscar información en tiempo real, ejecutar código, o decidir que necesita más datos antes de responder. Eso requiere un agente.

### 2.4 Mapa del Ecosistema: Dónde encaja cada framework

En 2025-2026 el ecosistema de agentes tiene múltiples frameworks con filosofías distintas. Elegir mal al inicio cuesta semanas de refactoring.

| Framework | Filosofía central | Fortaleza | Limitación | Cuándo elegir |
|---|---|---|---|---|
| **LangChain AgentExecutor** | Cadena + tools | Madurez, comunidad, integraciones | Verboso, ciclo ReAct fijo | Primer agente, integraciones rápidas |
| **LangGraph** | Grafo de estados con ciclos | Control total del flujo, HITL | Más código, curva de aprendizaje | Workflows complejos, estado persistente |
| **CrewAI** | Equipo de agentes con roles | Legible, role-based, memoria integrada | Menos control sobre el grafo | Pipelines de investigación con roles claros |
| **Google ADK** | Agentes corporativos + A2A | Protocolo estándar Google, multi-agente | Ecosistema Google-centrico | Enterprise, integración con Vertex AI |
| **Smolagents** | Código como acción | Flexible, composición arbitraria | Mayor superficie de seguridad | Análisis ad-hoc, tasks no predecibles |
| **AutoGen** (Microsoft) | Conversación multi-agente | Patrones de debate y crítica | Menos control, verbosidad alta | Revisión por pares, simulación de equipos |
| **MetaGPT** | Software company simulada | Genera código + docs completos | Muy lento, costoso | Generación de proyectos completos |

**Regla práctica para investigación:**
- Necesitas **explorar**: Smolagents CodeAgent
- Necesitas **pipeline reproducible**: LangGraph + CrewAI
- Necesitas **equipo multi-agente con roles**: CrewAI
- Necesitas **integrar con Google Cloud**: ADK
- Necesitas **revisión crítica entre agentes**: AutoGen

**Esta unidad cubre** los primeros cinco. AutoGen y MetaGPT quedan como extensión — comparten los mismos principios ReAct pero con patrones conversacionales distintos.


---
## 3. ReAct: el Paradigma Dominante <a id='3-react'></a>

### 3.1 Razonamiento + Actuación

ReAct (*Reasoning + Acting*, Yao et al., 2022) es el patrón de agente más influyente de la última década en IA aplicada. La idea central es simple pero poderosa: **intercalar razonamiento en lenguaje natural con acciones concretas**, en lugar de razonar primero y actuar después.

El ciclo ReAct por turno:

$$
\text{ReAct}(o_t) = \underbrace{r_t}_{\text{thought}} \rightarrow \underbrace{a_t}_{\text{action}} \rightarrow \underbrace{obs_t}_{\text{observation}} \rightarrow \text{ReAct}(o_{t+1})
$$

Cada iteración produce:
1. **Thought** (pensamiento): el agente razona sobre qué necesita hacer
2. **Action** (acción): llama a una tool específica con argumentos
3. **Observation** (observación): recibe el resultado de la tool
4. → Vuelve al paso 1 con nueva información

```
Usuario: "¿Cuál es la capitalización de mercado de Apple?"

Thought: Necesito datos financieros actuales. Usaré la tool de búsqueda.
Action: search_web("Apple market cap 2026")
Observation: "Apple AAPL: Market cap $3.2 trillion (Mar 2026)"

Thought: Tengo la información. Puedo responder.
Final Answer: La capitalización de mercado de Apple es aproximadamente $3.2 billones.
```

**¿Por qué es superior a cadenas fijas?** Porque el agente puede:
- Decidir cuántas veces necesita llamar a tools
- Elegir qué tool usar según el resultado de la anterior
- Detenerse cuando tiene suficiente información
- Pedir aclaración al usuario si los datos son insuficientes

### 3.2 Function Calling vs Tool Use

Estos términos se usan indistintamente pero tienen origen diferente:

| Término | Origen | Lo que hace |
|---------|--------|-------------|
| **Function Calling** | OpenAI (GPT-4) | El LLM genera JSON estructurado que describe qué función llamar y con qué argumentos. El programador ejecuta la función y devuelve el resultado. |
| **Tool Use** | Anthropic (Claude) | Equivalente conceptual, misma mecánica, diferente nombre. |
| **Tools en LangChain** | LangChain | Abstracción unificada sobre function calling y tool use que funciona con cualquier LLM compatible. |

En LangChain usaremos siempre la abstracción `@tool` — funciona igual con Gemini, GPT-4o y Claude.

### 3.3 Implementación con LangChain

In [4]:
# ============================================================
# WARM-UP: Sección 3 — Agentes y Tools
# ============================================================
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

print("Imports para agentes cargados correctamente.")

Imports para agentes cargados correctamente.


In [5]:
# ────────────────────────────────────────────────────────────
# Output esperado: agente ReAct con 2 tools usando tool calling
# ────────────────────────────────────────────────────────────

# Paso 1: Definir tools con el decorador @tool
# El docstring ES la descripción que ve el LLM — debe ser claro
@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática simple. 
    Úsala cuando necesites hacer cálculos numéricos.
    Ejemplo de input: '2 + 2', '100 * 0.15', '(5 ** 2) / 3'
    """
    # Paso 1a: Evaluador aritmético seguro con AST (sin eval() ni exec())
    import ast, operator as op
    _OPS = {
        ast.Add: op.add, ast.Sub: op.sub,
        ast.Mult: op.mul, ast.Div: op.truediv,
        ast.Pow: op.pow, ast.USub: op.neg,
    }
    def _safe_eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        elif isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
        elif isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_safe_eval(node.operand))
        else:
            raise ValueError(f"Operacion no permitida: {type(node).__name__}")
    try:
        tree = ast.parse(expresion, mode='eval')
        result = _safe_eval(tree.body)
        return str(result)
    except Exception as e:
        return f"Error al calcular: {e}"


@tool
def buscar_concepto(termino: str) -> str:
    """Busca la definición de un concepto técnico de IA o ciencia.
    Úsala cuando el usuario pregunte por el significado de un término.
    Ejemplo de input: 'transformer', 'embedding', 'gradient descent'
    """
    # Paso 2: Simulación de base de conocimiento local
    # En producción esto consultaría un vector store o API externa
    knowledge_base = {
        "transformer": "Arquitectura de red neuronal basada en mecanismos de atención (Vaswani et al., 2017). Base de todos los LLMs modernos.",
        "embedding": "Representación vectorial densa de texto, imágenes u otros datos en un espacio continuo de alta dimensión.",
        "gradient descent": "Algoritmo de optimización que ajusta parámetros de un modelo en la dirección opuesta al gradiente de la función de pérdida.",
        "langchain": "Framework de Python para construir aplicaciones con LLMs, incluyendo cadenas, agentes y RAG.",
        "react": "Patrón de agente que intercala razonamiento (thought) con acciones (action) en ciclos iterativos hasta obtener una respuesta final.",
        "rag": "Retrieval-Augmented Generation: técnica que combina recuperación de documentos relevantes con generación de texto por un LLM.",
    }
    termino_lower = termino.lower().strip()
    result = knowledge_base.get(termino_lower)
    if result:
        return result
    # Búsqueda parcial
    for key, val in knowledge_base.items():
        if termino_lower in key or key in termino_lower:
            return val
    return f"Concepto '{termino}' no encontrado en la base de conocimiento local."


# Paso 3: Crear el prompt del agente
# MessagesPlaceholder es donde el agente insertará el historial de tool calls
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Eres un asistente técnico de IA. Usa las tools disponibles cuando necesites "
     "calcular algo o buscar definiciones. Siempre explica tu razonamiento."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),  # historial de tool calls
])

# Paso 4: Crear el agente con tool calling
tools = [calcular, buscar_concepto]
agent = create_tool_calling_agent(llm, tools, prompt)

# Paso 5: AgentExecutor — wrapper que ejecuta el ciclo ReAct
# max_iterations es OBLIGATORIO para evitar loops infinitos
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,       # muestra el pensamiento y tool calls en consola
    max_iterations=5,   # límite de seguridad
    handle_parsing_errors=True,
)

print("Agente creado con tools:", [t.name for t in tools])

Agente creado con tools: ['calcular', 'buscar_concepto']


In [6]:
# Paso 6: Ejecutar el agente con diferentes tipos de consultas

# Consulta que requiere tool de cálculo
print("=" * 60)
print("CONSULTA 1: Requiere cálculo")
print("=" * 60)
result1 = agent_executor.invoke({
    "input": "Si tengo un presupuesto de $500 y el costo por 1M tokens es $0.10, ¿cuántos millones de tokens puedo usar?"
})
print("\nRespuesta final:", result1["output"])

CONSULTA 1: Requiere cálculo


> Entering new AgentExecutor chain...

Invoking: `calcular` with `{'expresion': '500 / 0.10'}`
responded: Para calcular cuántos millones de tokens puedes usar, necesito dividir tu presupuesto total entre el costo por 1 millón de tokens.

Presupuesto: $500
Costo por 1M tokens: $0.10

La operación sería: 500 / 0.10



5000.0Con un presupuesto de $500 y un costo de $0.10 por 1 millón de tokens, puedes usar 5000 millones de tokens.

> Finished chain.

Respuesta final: Con un presupuesto de $500 y un costo de $0.10 por 1 millón de tokens, puedes usar 5000 millones de tokens.


In [7]:
# Consulta que requiere búsqueda de concepto
print("=" * 60)
print("CONSULTA 2: Requiere búsqueda de concepto")
print("=" * 60)
result2 = agent_executor.invoke({
    "input": "¿Qué es RAG y cómo se diferencia de un agente ReAct?"
})
print("\nRespuesta final:", result2["output"])

CONSULTA 2: Requiere búsqueda de concepto


> Entering new AgentExecutor chain...

Invoking: `buscar_concepto` with `{'termino': 'RAG'}`


Retrieval-Augmented Generation: técnica que combina recuperación de documentos relevantes con generación de texto por un LLM.
Invoking: `buscar_concepto` with `{'termino': 'ReAct'}`


Patrón de agente que intercala razonamiento (thought) con acciones (action) en ciclos iterativos hasta obtener una respuesta final.RAG (Retrieval-Augmented Generation) es una técnica que mejora la capacidad de los modelos de lenguaje grandes (LLM) para generar respuestas informadas. Lo hace combinando la recuperación de información relevante de una base de datos o conjunto de documentos con la generación de texto. En esencia, el LLM primero busca información pertinente y luego utiliza esa información para formular su respuesta.

Por otro lado, ReAct (Reasoning and Acting) es un patrón de agente que permite a los LLM interactuar con el mundo exterior de una manera más d

**Output esperado:** Para la consulta 1, el agente debería mostrar un `Thought` sobre la necesidad de calcular, llamar a la tool `calcular` con `'500 / 0.10'`, recibir `5000.0`, y responder '5000 millones de tokens'. Para la consulta 2, debería llamar a `buscar_concepto` dos veces (una para RAG, una para ReAct) y sintetizar una comparativa.

**Interpretación:** El traza `verbose=True` es la representación directa del ciclo Percibir-Razonar-Actuar en código. Cada línea `Thought:` es la fase de razonamiento. Cada `Action:` con sus argumentos es la fase de actuación. Cada `Observation:` es la fase de percepción del resultado. Este ciclo puede repetirse N veces hasta que el agente decide que tiene suficiente información — en cadenas fijas esto es imposible.

**Errores comunes en esta sección:**
- Omitir `max_iterations` → el agente puede entrar en un bucle usando la misma tool repetidamente
- Docstrings vagos en los tools (ej: "hace cosas") → el LLM no sabrá cuándo usar el tool
- Hardcodear API keys en el código → siempre `os.environ.get("CLAVE")`

In [8]:
# ============================================================
# Tests unitarios — AST evaluator seguro + skill warm-up
# Se ejecutan sin dependencias externas (stdlib solo)
# Adapta los dominios de ejemplo a tu area de investigacion
# ============================================================
import ast, operator as op

# ── Replicar _safe_eval para testing independiente ──
_OPS = {
    ast.Add: op.add, ast.Sub: op.sub,
    ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}
def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    elif isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    elif isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    else:
        raise ValueError(f"Operacion no permitida: {type(node).__name__}")

def safe_calc(expr):
    tree = ast.parse(expr, mode='eval')
    return _safe_eval(tree.body)

# ── Test suite ──────────────────────────────────────────────
tests_passed = 0
tests_failed = 0

def check(label, got, expected, tol=1e-9):
    global tests_passed, tests_failed
    ok = abs(got - expected) < tol if isinstance(expected, float) else got == expected
    status = "PASS" if ok else "FAIL"
    if ok:
        tests_passed += 1
    else:
        tests_failed += 1
    print(f"  [{status}] {label}: got={got}, expected={expected}")

def check_error(label, expr):
    global tests_passed, tests_failed
    try:
        safe_calc(expr)
        print(f"  [FAIL] {label}: expected error but got result")
        tests_failed += 1
    except (ValueError, ZeroDivisionError):
        print(f"  [PASS] {label}: correctly raised error")
        tests_passed += 1

print("=== Tests: AST evaluator seguro ===")
check("suma simple",        safe_calc("2 + 2"),         4)
check("multiplicacion",     safe_calc("100 * 0.15"),    15.0, tol=1e-6)
check("parentesis",         safe_calc("(5 ** 2) / 3"),  25/3, tol=1e-6)
check("negativo",           safe_calc("-7 + 10"),        3)
check("cadena de ops",      safe_calc("1 + 2 * 3"),      7)   # precedencia correcta

print("\n=== Tests: bloqueo de operaciones inseguras ===")
check_error("import en expr",    "__import__('os')")
check_error("atributo",          "x.y")
check_error("llamada func",      "print('hola')")

print("\n=== Tests: skill warm-up (context_loader) ===")
try:
    from external_skills.agent_warmup.context_loader import warm_up_agent, SKILL_METADATA
    assert "name" in SKILL_METADATA,          "SKILL_METADATA debe tener 'name'"
    assert "version" in SKILL_METADATA,       "SKILL_METADATA debe tener 'version'"
    assert callable(warm_up_agent),           "warm_up_agent debe ser callable"
    # Verifica que acepta dominios arbitrarios (no solo nano)
    result = warm_up_agent("research")
    assert isinstance(result, str) and len(result) > 0, "warm_up_agent debe retornar string"
    print("  [PASS] context_loader: SKILL_METADATA valido y warm_up_agent callable")
    tests_passed += 1
except ImportError:
    print("  [SKIP] context_loader no disponible en este entorno")
except AssertionError as e:
    print(f"  [FAIL] context_loader: {e}")
    tests_failed += 1

print(f"\n{'='*50}")
print(f"Resultado: {tests_passed} PASS, {tests_failed} FAIL")
if tests_failed == 0:
    print("Todos los tests pasaron. El evaluador AST es seguro.")
else:
    print("Revisar los tests fallidos antes de usar en produccion.")


=== Tests: AST evaluator seguro ===
  [PASS] suma simple: got=4, expected=4
  [PASS] multiplicacion: got=15.0, expected=15.0
  [PASS] parentesis: got=8.333333333333334, expected=8.333333333333334
  [PASS] negativo: got=3, expected=3
  [PASS] cadena de ops: got=7, expected=7

=== Tests: bloqueo de operaciones inseguras ===
  [PASS] import en expr: correctly raised error
  [PASS] atributo: correctly raised error
  [PASS] llamada func: correctly raised error

=== Tests: skill warm-up (context_loader) ===
  [SKIP] context_loader no disponible en este entorno

Resultado: 8 PASS, 0 FAIL
Todos los tests pasaron. El evaluador AST es seguro.


**Por qué testear el evaluador AST:**

`eval()` con `__builtins__: {}` sigue siendo peligroso — un atacante puede
acceder a clases base via `().__class__.__bases__`. El parser AST elimina esa
superficie de ataque completamente: solo evalúa nodos `Constant`, `BinOp` y
`UnaryOp`. Los tests de arriba verifican tanto el comportamiento correcto como
el bloqueo de expresiones maliciosas.

**Aplicabilidad:** Este patrón de evaluador seguro es reutilizable en cualquier
sistema multi-agente que necesite ejecutar expresiones aritméticas provistas
por usuarios — física, economía, bioinformática, ingeniería.


In [9]:
# ============================================================
# Tests unitarios — AST evaluator seguro + skill warm-up
# Se ejecutan sin dependencias externas (stdlib solo)
# Adapta los dominios de ejemplo a tu area de investigacion
# ============================================================
import ast, operator as op

# ── Replicar _safe_eval para testing independiente ──
_OPS = {
    ast.Add: op.add, ast.Sub: op.sub,
    ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}
def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    elif isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    elif isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    else:
        raise ValueError(f"Operacion no permitida: {type(node).__name__}")

def safe_calc(expr):
    tree = ast.parse(expr, mode='eval')
    return _safe_eval(tree.body)

# ── Test suite ──────────────────────────────────────────────
tests_passed = 0
tests_failed = 0

def check(label, got, expected, tol=1e-9):
    global tests_passed, tests_failed
    ok = abs(got - expected) < tol if isinstance(expected, float) else got == expected
    status = "PASS" if ok else "FAIL"
    if ok:
        tests_passed += 1
    else:
        tests_failed += 1
    print(f"  [{status}] {label}: got={got}, expected={expected}")

def check_error(label, expr):
    global tests_passed, tests_failed
    try:
        safe_calc(expr)
        print(f"  [FAIL] {label}: expected error but got result")
        tests_failed += 1
    except (ValueError, ZeroDivisionError):
        print(f"  [PASS] {label}: correctly raised error")
        tests_passed += 1

print("=== Tests: AST evaluator seguro ===")
check("suma simple",        safe_calc("2 + 2"),         4)
check("multiplicacion",     safe_calc("100 * 0.15"),    15.0, tol=1e-6)
check("parentesis",         safe_calc("(5 ** 2) / 3"),  25/3, tol=1e-6)
check("negativo",           safe_calc("-7 + 10"),        3)
check("cadena de ops",      safe_calc("1 + 2 * 3"),      7)   # precedencia correcta

print("\n=== Tests: bloqueo de operaciones inseguras ===")
check_error("import en expr",    "__import__('os')")
check_error("atributo",          "x.y")
check_error("llamada func",      "print('hola')")

print("\n=== Tests: skill warm-up (context_loader) ===")
try:
    from external_skills.agent_warmup.context_loader import warm_up_agent, SKILL_METADATA
    assert "name" in SKILL_METADATA,          "SKILL_METADATA debe tener 'name'"
    assert "version" in SKILL_METADATA,       "SKILL_METADATA debe tener 'version'"
    assert callable(warm_up_agent),           "warm_up_agent debe ser callable"
    # Verifica que acepta dominios arbitrarios (no solo nano)
    result = warm_up_agent("research")
    assert isinstance(result, str) and len(result) > 0, "warm_up_agent debe retornar string"
    print("  [PASS] context_loader: SKILL_METADATA valido y warm_up_agent callable")
    tests_passed += 1
except ImportError:
    print("  [SKIP] context_loader no disponible en este entorno")
except AssertionError as e:
    print(f"  [FAIL] context_loader: {e}")
    tests_failed += 1

print(f"\n{'='*50}")
print(f"Resultado: {tests_passed} PASS, {tests_failed} FAIL")
if tests_failed == 0:
    print("Todos los tests pasaron. El evaluador AST es seguro.")
else:
    print("Revisar los tests fallidos antes de usar en produccion.")


=== Tests: AST evaluator seguro ===
  [PASS] suma simple: got=4, expected=4
  [PASS] multiplicacion: got=15.0, expected=15.0
  [PASS] parentesis: got=8.333333333333334, expected=8.333333333333334
  [PASS] negativo: got=3, expected=3
  [PASS] cadena de ops: got=7, expected=7

=== Tests: bloqueo de operaciones inseguras ===
  [PASS] import en expr: correctly raised error
  [PASS] atributo: correctly raised error
  [PASS] llamada func: correctly raised error

=== Tests: skill warm-up (context_loader) ===
  [SKIP] context_loader no disponible en este entorno

Resultado: 8 PASS, 0 FAIL
Todos los tests pasaron. El evaluador AST es seguro.


**Por qué testear el evaluador AST:**

`eval()` con `__builtins__: {}` sigue siendo peligroso — un atacante puede
acceder a clases base via `().__class__.__bases__`. El parser AST elimina esa
superficie de ataque completamente: solo evalúa nodos `Constant`, `BinOp` y
`UnaryOp`. Los tests de arriba verifican tanto el comportamiento correcto como
el bloqueo de expresiones maliciosas.

**Aplicabilidad:** Este patrón de evaluador seguro es reutilizable en cualquier
sistema multi-agente que necesite ejecutar expresiones aritméticas provistas
por usuarios — física, economía, bioinformática, ingeniería.


---
## 4. Skills y Warm-Ups <a id='4-skills'></a>

### 4.1 ¿Qué es una Skill?

Una de las confusiones más frecuentes al aprender sobre agentes es mezclar tres conceptos relacionados pero distintos:

| Concepto | Definición | Scope | Ejemplo en este proyecto |
|----------|-----------|-------|-------------------------|
| **Tool** | Función que el LLM puede *llamar* en tiempo de ejecución para obtener datos o ejecutar acciones | Una función concreta | `calcular()`, `buscar_concepto()` |
| **Skill** | Módulo que *inyecta* capacidad especializada en un agente antes de que ejecute (warm-up) | Sistema prompt + tools relacionadas + memoria de dominio | `context_loader.py` |
| **Chain/Graph** | Orquestación del *flujo* entre agentes y nodos | Arquitectura completa | `LangGraph StateGraph` |

La metáfora: si el agente es un ingeniero recién contratado, las **tools** son su caja de herramientas y las **skills** son el *onboarding* que le explica el contexto de la empresa antes de su primer día.

Una skill puede:
1. Inyectar **conocimiento de dominio** en el system prompt (lo que el agente sabe antes de empezar)
2. Registrar **tools especializadas** adicionales
3. Pre-cargar **memoria de trabajo** (episodios pasados relevantes)
4. Configurar **parámetros** específicos del dominio (temperatura, max tokens, etc.)

```
Agente base  +  Skill de investigación  =  Agente investigador
Agente base  +  Skill legal             =  Agente revisor de contratos
Agente base  +  Skill nanotecnología    =  Agente de simulación molecular
```

### 4.2 El Patrón Warm-Up

El **warm-up** es el proceso de inicialización de una skill. Ocurre **antes** de que el agente ejecute su primera tarea y su propósito es configurar el contexto óptimo para el dominio específico.

Analogía con deportes: un atleta no empieza a correr desde el sofá. Tiene una rutina de calentamiento que prepara músculos y mente para la competición. Un agente bien diseñado hace lo mismo.

```
SIN warm-up:    [Task llega] → [Agente responde desde contexto genérico]
CON warm-up:    [Skill.warm_up()] → [Contexto especializado cargado] →
                [Task llega] → [Agente responde con expertise de dominio]
```

### 4.3 Skill Registry

En sistemas multi-agente, múltiples agentes pueden necesitar la misma skill. El **Skill Registry** es un catálogo centralizado donde:
- Los agentes descubren qué skills están disponibles
- Las skills se cargan con un nombre en lugar de un import path
- Se mantiene versionado y compatibilidad

### 4.4 Construir `context_loader.py` paso a paso

In [10]:
# ============================================================
# WARM-UP: Sección 4 — Skills y Registry
# ============================================================
import sys
from pathlib import Path

# Resolver el path raíz del proyecto para importar external_skills
# La notebook está en: educational_content/unit_05_multi_agent_sys/
# El proyecto raíz está 3 niveles arriba
project_root = Path.cwd()

# Buscar la carpeta external_skills subiendo niveles
for _ in range(5):
    if (project_root / "external_skills").exists():
        break
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root detectado: {project_root}")
print(f"external_skills existe: {(project_root / 'external_skills').exists()}")

Project root detectado: d:\Users\UCEMICH\Desktop\antigravity projects\IA NANOTECNOLOGIA\Antigravity-Nano-Research-Multiagentic-Core
external_skills existe: True


In [11]:
# ────────────────────────────────────────────────────────────
# Output esperado: usar la skill context_loader en un agente
# ────────────────────────────────────────────────────────────

# Paso 1: Cargar la skill
from external_skills.agent_warmup.context_loader import warm_up, list_available_domains, SKILL_METADATA

print("Skill cargada:", SKILL_METADATA["name"], "v", SKILL_METADATA["version"])
print("Dominios disponibles:", list_available_domains())

Skill cargada: context_loader v 1.0.0
Dominios disponibles: ['research', 'engineering', 'data_analysis', 'teaching', 'nanotechnology', 'custom']


In [12]:
# Paso 2: Ejecutar el warm-up para el dominio de investigación
context = warm_up(domain="research")

print("Dominio activado:", context["domain"])
print("\nContexto inyectado al sistema:")
print("-" * 40)
print(context["system_context"])
print("-" * 40)
print(f"\nMensajes listos para LangChain: {len(context['messages'])} mensaje(s)")

Dominio activado: research

Contexto inyectado al sistema:
----------------------------------------
Eres un asistente de investigación científica especializado en análisis de literatura, síntesis de conocimiento y generación de hipótesis. Aportas rigor metodológico y citas evidencia cuando es posible. Tu audiencia es investigadores de postgrado.
----------------------------------------

Mensajes listos para LangChain: 1 mensaje(s)


In [13]:
# Paso 3: Comparar un agente SIN y CON warm-up de skill
from langchain_core.messages import HumanMessage, SystemMessage

pregunta = "¿Cómo evalúo la solidez de un paper científico que encontré?"

# Agente SIN warm-up (contexto genérico)
response_sin_warmup = llm.invoke([HumanMessage(content=pregunta)])

# Agente CON warm-up de la skill (contexto de investigador senior)
response_con_warmup = llm.invoke(
    context["messages"] + [HumanMessage(content=pregunta)]
)

print("SIN warm-up (respuesta genérica):")
print("-" * 40)
print(response_sin_warmup.content[:400], "...")

print("\nCON warm-up (contexto: investigador senior):")
print("-" * 40)
print(response_con_warmup.content[:400], "...")

SIN warm-up (respuesta genérica):
----------------------------------------
Evaluar la solidez de un paper científico es una habilidad crucial para cualquier persona que consuma información científica. Aquí te presento una guía completa con los aspectos clave a considerar, dividida en secciones para facilitar su comprensión:

## I. Evaluación Inicial y General

1.  **Título y Resumen (Abstract):**
    *   **Claridad y Concisión:** ¿El título es informativo y el resumen pr ...

CON warm-up (contexto: investigador senior):
----------------------------------------
Evaluar la solidez de un *paper* científico es una habilidad crítica para cualquier investigador de posgrado. No se trata solo de leer el resumen, sino de un análisis sistemático y crítico de cada sección. A continuación, te presento un marco de evaluación basado en principios de rigor metodológico y buenas prácticas científicas, citando la evidencia cuando es pertinente.

### Marco para Evaluar l ...


**Output esperado:** La respuesta CON warm-up debe ser notablemente más específica y técnica: debe mencionar metodología, validez estadística, potencial de sesgo, peer review, reproducibilidad — los criterios que un investigador senior realmente usa. La respuesta sin warm-up será más genérica.

**Interpretación:** Este experimento muestra directamente el valor de las skills. El agente usa el mismo LLM en ambos casos — la diferencia es únicamente el contexto de sistema inyectado por `warm_up()`. La skill transforma un asistente genérico en un investigador científico especializado sin cambiar una sola línea del código del agente. Este patrón es fundamental para construir sistemas multi-agente donde cada agente tiene una especialidad clara.

In [14]:
# Paso 4: Registrar la skill en el Skill Registry del proyecto
# Esto permite que otros agentes la descubran dinámicamente

# Verificar que el registry existe; si no, mostramos el formato de registro
registry_path = project_root / "external_skills" / "registry.py"

REGISTRY_ENTRY = {
    "agent_warmup.context_loader": {
        "module": "external_skills.agent_warmup.context_loader",
        "description": SKILL_METADATA["description"],
        "input": SKILL_METADATA["input"],
        "output": SKILL_METADATA["output"],
        "domains": SKILL_METADATA["domain"],
        "version": SKILL_METADATA["version"],
    }
}

print("Entrada de registry lista:")
import json
print(json.dumps(REGISTRY_ENTRY, indent=2))

if registry_path.exists():
    print(f"\nRegistry encontrado en: {registry_path}")
    print("La skill ya fue registrada. Verificar con: load_skill('agent_warmup.context_loader')")
else:
    print(f"\nRegistry no encontrado en {registry_path}.")
    print("Ver sección 4.3 del plan para crear external_skills/registry.py")

Entrada de registry lista:
{
  "agent_warmup.context_loader": {
    "module": "external_skills.agent_warmup.context_loader",
    "description": "Inyecta contexto de dominio en agentes LLM. Soporta 5 dominios predefinidos y contexto custom.",
    "input": "domain: str, custom_config: dict | None",
    "output": "dict con system_context, messages, domain",
    "domains": "agent_warmup",
    "version": "1.0.0"
  }
}

Registry encontrado en: d:\Users\UCEMICH\Desktop\antigravity projects\IA NANOTECNOLOGIA\Antigravity-Nano-Research-Multiagentic-Core\external_skills\registry.py
La skill ya fue registrada. Verificar con: load_skill('agent_warmup.context_loader')


---
## 5. LangChain LCEL: Composición de Cadenas <a id='5-lcel'></a>

### 5.1 El Operador `|`

**LCEL** (LangChain Expression Language) es el sistema de composición de LangChain desde la versión 0.1. Usa el operador `|` (pipe) para encadenar componentes, igual que las pipes de Unix.

La regla de composición es simple: el output de un componente se convierte en el input del siguiente.

```python
cadena = prompt | llm | output_parser
#         ↑         ↑        ↑
#      str→Msg   Msg→Msg   Msg→str
```

Todo componente LCEL implementa la interfaz **Runnable** con métodos `.invoke()`, `.stream()`, `.batch()`, y `.ainvoke()` (async). Esto significa que puedes intercambiar cualquier componente por otro que tenga la misma firma de input/output.

### 5.2 Agente con Tools usando LCEL

In [15]:
# ────────────────────────────────────────────────────────────
# Output esperado: agente con skill warm-up integrado via LCEL
# ────────────────────────────────────────────────────────────
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

# Paso 1: Cargar el warm-up de la skill de ingeniería
engineering_context = warm_up(domain="engineering")

# Paso 2: Construir el prompt con el contexto de la skill inyectado
prompt_con_skill = ChatPromptTemplate.from_messages([
    # El system message viene de la skill — no lo escribimos a mano
    ("system", engineering_context["system_context"]),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Paso 3: Tools para el agente de ingeniería
@tool
def revisar_codigo(codigo: str) -> str:
    """Analiza un fragmento de código Python e identifica problemas potenciales.
    Úsala cuando el usuario presente código para revisión.
    Input: código Python como string.
    """
    issues = []
    if "except:" in codigo and "except Exception" not in codigo:
        issues.append("Uso de except desnudo — captura demasiado amplia")
    if "print(" in codigo and "logging" not in codigo:
        issues.append("Usar logging en lugar de print para código de producción")
    if "import *" in codigo:
        issues.append("Evitar 'import *' — hace el namespace opaco")
    if not issues:
        return "No se detectaron problemas obvios de estilo en el fragmento."
    return "Problemas detectados:\n" + "\n".join(f"- {i}" for i in issues)

tools_eng = [calcular, revisar_codigo]  # reutilizamos calcular de sección 3

# Paso 4: Crear agente con la skill inyectada
agent_eng = create_tool_calling_agent(llm, tools_eng, prompt_con_skill)
executor_eng = AgentExecutor(
    agent=agent_eng,
    tools=tools_eng,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
)

print("Agente de ingeniería creado con skill 'engineering' + tools:",
      [t.name for t in tools_eng])

Agente de ingeniería creado con skill 'engineering' + tools: ['calcular', 'revisar_codigo']


In [16]:
# Paso 5: Invocar el agente con una tarea de desarrollo
result_eng = executor_eng.invoke({
    "input": """
    Revisa este código y dime si tiene problemas:
    
    def procesar_datos(datos):
        try:
            resultado = []
            for item in datos:
                if item > 100:
                    resultado.append(item * 2)
            print(f'Procesados {len(resultado)} items')
            return resultado
        except:
            print('Error al procesar')
            return []
    """
})
print("\nRespuesta del agente de ingeniería:")
print(result_eng["output"])



> Entering new AgentExecutor chain...

Invoking: `revisar_codigo` with `{'codigo': "\n    def procesar_datos(datos):\n        try:\n            resultado = []\n            for item in datos:\n                if item > 100:\n                    resultado.append(item * 2)\n            print(f'Procesados {len(resultado)} items')\n            return resultado\n        except:\n            print('Error al procesar')\n            return []"}`


Problemas detectados:
- Uso de except desnudo — captura demasiado amplia
- Usar logging en lugar de print para código de producciónEl código tiene un `except` desnudo, lo cual es una mala práctica porque captura todas las excepciones, incluyendo las que no esperas y que podrían ocultar errores de programación. Es mejor especificar las excepciones que esperas manejar.

Además, para aplicaciones en producción, se recomienda usar un sistema de logging en lugar de `print` para los mensajes de depuración y error. Esto permite un control más granular sobr

**Interpretación:** El agente debería llamar a `revisar_codigo` con el fragmento, recibir los 2 problemas detectados (`except:` desnudo y `print`), y luego con su contexto de skill de ingeniería elaborar recomendaciones detalladas sobre cómo corregirlos siguiendo SOLID/DRY. La skill de engineering hace que las recomendaciones sean más técnicas y específicas que sin ella.

---
## 6. Los 4 Tipos de Memoria en Agentes <a id='6-memoria'></a>

La memoria es uno de los aspectos más subestimados del diseño de agentes. Un agente sin gestión de memoria adecuada es como un profesional con amnesia: puede ser brillante en el momento, pero no aprende de sesiones pasadas ni mantiene coherencia a largo plazo.

La taxonomía proviene de MemGPT/Letta y la literatura cognitiva aplicada a LLMs:

| Tipo | ¿Qué almacena? | Duración | Herramienta típica | Analogía |
|------|---------------|----------|--------------------|----------|
| **Sensorial / In-Context** | El historial activo de conversación, el prompt actual | Solo mientras está en la ventana del LLM | `ConversationBufferMemory`, mensajes en state de LangGraph | RAM de computadora |
| **Episódica** | Eventos y experiencias pasadas con timestamps | Persistente entre sesiones | Mem0, Zep, PostgreSQL con timestamps | Diario personal |
| **Semántica** | Conocimiento factual, documentos, hechos | Persistente | ChromaDB, Pinecone, RAG sobre vector store | Enciclopedia |
| **Procedimental** | Cómo hacer las cosas: skills, prompts, fine-tuning | Persistente (en código/pesos) | Skill Registry, few-shot examples, LoRA | Memoria muscular |

**La regla de oro de la memoria:** cargar solo la memoria necesaria para la tarea actual. Cargar toda la memoria disponible desperdicia tokens y puede confundir al agente.

```
ESTRATEGIA DE WARM-UP con memoria:

1. [Semántica] RAG query → documentos relevantes para ESTA tarea
2. [Episódica] Recuperar las 3-5 interacciones pasadas más relevantes
3. [Procedimental] Cargar skills del dominio de la tarea
4. [Sensorial] Todo lo anterior entra al context window
      ↓
   [TAREA] → agente responde con memoria óptima cargada
```

In [17]:
# ────────────────────────────────────────────────────────────
# Output esperado: demostración práctica de memoria in-context
# ────────────────────────────────────────────────────────────
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain.chains import ConversationChain

# Paso 1: Memoria in-context — guarda TODOS los mensajes
memory_buffer = ConversationBufferMemory(return_messages=True)

# Paso 2: Cadena con memoria buffer
conv_chain = ConversationChain(
    llm=llm,
    memory=memory_buffer,
    verbose=False
)

# Simular una conversación multi-turno
turns = [
    "Mi nombre es Ana y estudio nanotecnología.",
    "¿Qué es un embedding y para qué se usa?",
    "¿Recuerdas lo que te dije sobre mi área de estudio?",  # prueba de memoria
]

for i, msg in enumerate(turns, 1):
    response = conv_chain.predict(input=msg)
    print(f"\nTurno {i}")
    print(f"Usuario: {msg}")
    print(f"Agente: {response[:200]}..." if len(response) > 200 else f"Agente: {response}")

# Inspeccionar cuántos tokens estamos usando en memoria
messages_en_memoria = memory_buffer.chat_memory.messages
print(f"\nMensajes almacenados en memoria in-context: {len(messages_en_memoria)}")

C:\Users\UCEMICH\AppData\Local\Temp\ipykernel_16052\4080137008.py:8: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory_buffer = ConversationBufferMemory(return_messages=True)
C:\Users\UCEMICH\AppData\Local\Temp\ipykernel_16052\4080137008.py:11: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :meth:`~RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html` instead.
  conv_chain = ConversationChain(



Turno 1
Usuario: Mi nombre es Ana y estudio nanotecnología.
Agente: ¡Hola, Ana! ¡Qué gusto conocerte! Soy un modelo de lenguaje grande, entrenado por Google. Es fascinante que estudies nanotecnología. ¿Podrías contarme un poco más sobre qué te atrajo a este campo tan ...

Turno 2
Usuario: ¿Qué es un embedding y para qué se usa?
Agente: ¡Excelente pregunta, Ana! Me encanta que me preguntes sobre temas tan fundamentales en el campo de la inteligencia artificial y el procesamiento del lenguaje natural.

Un **embedding** (o incrustación...

Turno 3
Usuario: ¿Recuerdas lo que te dije sobre mi área de estudio?
Agente: ¡Claro que sí, Ana! ¡Por supuesto que lo recuerdo! Me dijiste que **estudias nanotecnología**.

De hecho, fue lo primero que mencionaste al presentarte, y me pareció fascinante. Incluso te pregunté qu...

Mensajes almacenados en memoria in-context: 6


In [18]:
# Paso 3: El problema del buffer — crece sin límite
# Solución: ConversationSummaryMemory (comprime el historial antiguo)

memory_summary = ConversationSummaryMemory(
    llm=llm,
    return_messages=True
)

conv_chain_summary = ConversationChain(
    llm=llm,
    memory=memory_summary,
    verbose=False
)

# Misma conversación
for msg in turns:
    conv_chain_summary.predict(input=msg)

print("Memoria con resumen:")
print(memory_summary.buffer)
print(f"\nEl historial se comprimió a un resumen — ahorra tokens en conversaciones largas.")

C:\Users\UCEMICH\AppData\Local\Temp\ipykernel_16052\3735352118.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory_summary = ConversationSummaryMemory(


Memoria con resumen:
Current summary:
The human introduces herself as Ana, a nanotechnology student. The AI greets Ana, expresses interest in her field, and asks about her specific interests within nanotechnology (nanomedicine, nanomaterials, or nanoelectronics). The AI explains that while it doesn't "study" like a human, it has been trained on a vast amount of data including nanotechnology, defining it as the manipulation of matter at an atomic and molecular scale (1-100 nanometers) and listing its broad applications in energy, materials, medicine, and electronics. The AI then offers to explain or discuss any particular aspect of nanotechnology Ana might be interested in. Ana then asks what an "embedding" is and what it's used for. The AI explains that embeddings are fundamental to its operation as a language model, defining them as numerical representations (vectors of numbers) of words, phrases, or concepts that allow computers to understand and process them. It highlights that simi

**Interpretación:** En el turno 3 el agente con `ConversationBufferMemory` recuerda correctamente que Ana estudia nanotecnología — eso es memoria sensorial/in-context funcionando. Con `ConversationSummaryMemory`, el historial se comprime a un resumen que preserva los hechos clave (nombre, área de estudio) usando muchos menos tokens. Para conversaciones de 3 turnos la diferencia es pequeña, pero en conversaciones de 50+ turnos la diferencia puede ser de miles de tokens por llamada.

Los tipos episódico (Mem0/Zep), semántico (RAG/vectores) y procedimental (Skills Registry) se profundizan en U5_05 y U5_08.

---
## 7. Patrones de Razonamiento Avanzado <a id='7-patrones'></a>

### 7.1 Chain of Thought (CoT)

**Chain of Thought** (Wei et al., 2022) es la técnica de prompting que instruye al LLM a mostrar su proceso de razonamiento paso a paso antes de dar la respuesta final. Mejora significativamente el rendimiento en tareas de razonamiento matemático, lógico y multi-paso.

La diferencia con ReAct: CoT es solo razonamiento interno (thought), sin acciones ni tools. ReAct = CoT + tool calls.

### 7.2 Self-Reflection

**Self-Reflection** (Shinn et al., 2023) permite a un agente revisar y corregir sus propias respuestas. El patrón:

```
[Generar respuesta inicial] → [Evaluar la respuesta] → [Identificar errores] → [Regenerar mejorada]
```

In [19]:
# ────────────────────────────────────────────────────────────
# Output esperado: CoT y Self-Reflection en práctica
# ────────────────────────────────────────────────────────────
from langchain_core.prompts import PromptTemplate

# Paso 1: Chain of Thought con few-shot
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Responde paso a paso. Muestra tu razonamiento explícitamente antes de dar la respuesta final. "
     "Formato:\nPaso 1: [razonamiento]\nPaso 2: [razonamiento]\n...\nRespuesta: [respuesta final]"),
    ("human", "{pregunta}")
])

cot_chain = cot_prompt | llm | StrOutputParser()

result_cot = cot_chain.invoke({
    "pregunta": "Si un agente procesa 100 documentos y tarda 2 segundos por documento, "
                "pero puede procesar 5 en paralelo, ¿cuánto tiempo total tardará?"
})
print("CoT Response:")
print(result_cot)

CoT Response:
Paso 1: Calcular el tiempo total que tomaría procesar todos los documentos si se hicieran uno por uno.
Tiempo por documento = 2 segundos
Número total de documentos = 100
Tiempo total (secuencial) = 100 documentos * 2 segundos/documento = 200 segundos

Paso 2: Determinar cuántos grupos de documentos se pueden procesar en paralelo.
Capacidad de procesamiento en paralelo = 5 documentos
Número total de documentos = 100
Número de grupos = 100 documentos / 5 documentos/grupo = 20 grupos

Paso 3: Calcular el tiempo que tarda en procesarse cada grupo de documentos en paralelo.
Dado que se procesan 5 documentos en paralelo, y cada documento tarda 2 segundos, el tiempo que tarda en completarse un "lote" de 5 documentos es el tiempo que tarda el documento más lento en ese lote. En este caso, todos tardan lo mismo.
Tiempo por grupo = 2 segundos

Paso 4: Calcular el tiempo total multiplicando el número de grupos por el tiempo que tarda cada grupo.
Tiempo total = Número de grupos * Tie

In [20]:
# Paso 2: Self-Reflection — el agente revisa su propia respuesta
from langchain_core.output_parsers import StrOutputParser

# Cadena de generación inicial
generate_prompt = ChatPromptTemplate.from_template(
    "Responde la siguiente pregunta de forma concisa:\n{pregunta}"
)

# Cadena de reflexión
reflect_prompt = ChatPromptTemplate.from_template(
    """Respuesta original: {respuesta_original}

Reflexiona críticamente sobre esta respuesta:
1. ¿Qué está bien?
2. ¿Qué podría estar incompleto o incorrecto?
3. Proporciona una versión mejorada."""
)

pregunta_test = "¿Cuál es la diferencia entre un agente y una cadena en LangChain?"

# Paso 3: Ejecutar el ciclo de self-reflection
respuesta_inicial = (generate_prompt | llm | StrOutputParser()).invoke({"pregunta": pregunta_test})
print("Respuesta inicial:")
print(respuesta_inicial)
print()

respuesta_mejorada = (reflect_prompt | llm | StrOutputParser()).invoke({
    "respuesta_original": respuesta_inicial
})
print("Respuesta mejorada (tras self-reflection):")
print(respuesta_mejorada)

Respuesta inicial:
**Agente:** Toma decisiones, usa herramientas y razona para lograr un objetivo.
**Cadena:** Secuencia predefinida de pasos (LLM, herramientas, etc.) para una tarea específica.

Respuesta mejorada (tras self-reflection):
¡Excelente ejercicio de reflexión! Analicemos la respuesta original:

---

**Respuesta original:**
**Agente:** Toma decisiones, usa herramientas y razona para lograr un objetivo.
**Cadena:** Secuencia predefinida de pasos (LLM, herramientas, etc.) para una tarea específica.

---

### 1. ¿Qué está bien?

*   **Concisión y claridad:** Ambas definiciones son directas y fáciles de entender a primera vista.
*   **Identificación de características clave:**
    *   Para el **Agente**, se mencionan correctamente la toma de decisiones, el uso de herramientas y el razonamiento, que son sus pilares fundamentales.
    *   Para la **Cadena**, se destaca la idea de "secuencia predefinida de pasos" y la inclusión de LLMs y herramientas, lo cual es acertado.
*   **Di

**Interpretación:** La respuesta mejorada tras self-reflection debería ser más completa. El modelo identifica aspectos que omitió en la primera respuesta. Este patrón es especialmente útil en tareas de escritura, revisión de código y razonamiento complejo donde una sola pasada raramente produce el mejor resultado. Veremos una versión más elaborada de self-reflection implementada como nodo de grafo en U5_02 con LangGraph.

---
## 8. Proyecto: Asistente de Investigación con Warm-Up <a id='8-proyecto'></a>

Ahora integramos todo lo aprendido en un sistema funcional: un asistente de investigación que usa la skill `context_loader`, tiene 2 tools especializadas, y aplica Chain of Thought en sus respuestas.

In [21]:
# ============================================================
# PROYECTO: Asistente de Investigación con Skill Warm-Up
# ============================================================
from external_skills.agent_warmup.context_loader import warm_up
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Paso 1: Warm-up del dominio de investigación
research_context = warm_up(domain="research")

# Paso 2: Tools especializadas para investigación
@tool
def buscar_papers(tema: str, max_resultados: int = 3) -> str:
    """Busca papers científicos recientes sobre un tema dado.
    Úsala cuando necesites encontrar literatura académica relevante.
    Input: tema de búsqueda en inglés o español, número máximo de resultados.
    """
    # Simulación — en U5_05 conectaremos con arXiv API real
    papers_db = {
        "multi-agent": [
            "ReAct: Synergizing Reasoning and Acting (Yao et al., 2022) — arXiv:2210.03629",
            "AutoGen: Enabling Next-Gen LLM Applications (Wu et al., 2023) — arXiv:2308.08155",
            "AgentBench: Evaluating LLMs as Agents (Liu et al., 2023) — arXiv:2308.03688",
        ],
        "rag": [
            "Retrieval-Augmented Generation for Knowledge-Intensive NLP (Lewis et al., 2020)",
            "RAG vs Fine-tuning: Pipelines, Tradeoffs, and a Case Study (Ovadia et al., 2023)",
        ],
        "memory": [
            "MemGPT: Towards LLMs as Operating Systems (Packer et al., 2023)",
            "Cognitive Architectures for Language Agents (Sumers et al., 2023)",
        ],
        "langchain": [
            "Documentación oficial: https://python.langchain.com/docs/",
            "LangGraph: Building Stateful, Multi-Actor Applications (LangChain, 2024)",
        ]
    }
    tema_lower = tema.lower()
    for key, papers in papers_db.items():
        if key in tema_lower or tema_lower in key:
            return "\n".join(papers[:max_resultados])
    return f"No se encontraron papers sobre '{tema}' en la base local. Sugiero buscar en arXiv.org o Semantic Scholar."


@tool
def evaluar_metodologia(descripcion: str) -> str:
    """Evalúa la metodología descrita en un paper o experimento.
    Identifica fortalezas y posibles debilidades del diseño experimental.
    Input: descripción del método o experimento a evaluar.
    """
    # Esta tool delega al LLM con un prompt estructurado
    eval_prompt = f"""Evalúa esta metodología científica en 3 puntos:
1. Aspecto más sólido del diseño
2. Principal limitación potencial
3. Sugerencia de mejora

Metodología: {descripcion}"""
    return llm.invoke(eval_prompt).content

# Paso 3: Prompt con contexto de skill inyectado + CoT
research_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     research_context["system_context"] + "\n\n"
     "Cuando respondas, muestra tu proceso de razonamiento paso a paso."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Paso 4: Crear el agente investigador
research_tools = [buscar_papers, evaluar_metodologia, calcular]
research_agent = create_tool_calling_agent(llm, research_tools, research_prompt)
research_executor = AgentExecutor(
    agent=research_agent,
    tools=research_tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True,
)

print("Asistente de investigación listo.")
print("Skill activa: research (investigador senior)")
print("Tools:", [t.name for t in research_tools])

Asistente de investigación listo.
Skill activa: research (investigador senior)
Tools: ['buscar_papers', 'evaluar_metodologia', 'calcular']


In [22]:
# Paso 5: Ejecutar una consulta de investigación compleja
resultado_proyecto = research_executor.invoke({
    "input": """
    Necesito entender el estado del arte en sistemas multi-agente con LLMs.
    ¿Cuáles son los papers más importantes? 
    Si hay 3 papers principales y los autores de cada uno publicaron en promedio 
    2 papers de seguimiento, ¿cuántos papers de seguimiento existen en total?
    """
})

print("\n" + "="*60)
print("RESPUESTA FINAL DEL ASISTENTE:")
print("="*60)
print(resultado_proyecto["output"])



> Entering new AgentExecutor chain...

Invoking: `buscar_papers` with `{'max_resultados': 3, 'tema': 'multi-agent systems with LLMs'}`
responded: Para responder a tu pregunta, seguiré los siguientes pasos:

1.  **Buscar papers** sobre "sistemas multi-agente con LLMs" para identificar los trabajos más relevantes.
2.  **Calcular** el número total de papers de seguimiento basándome en la información proporcionada.

Primero, buscaré los papers:


ReAct: Synergizing Reasoning and Acting (Yao et al., 2022) — arXiv:2210.03629
AutoGen: Enabling Next-Gen LLM Applications (Wu et al., 2023) — arXiv:2308.08155
AgentBench: Evaluating LLMs as Agents (Liu et al., 2023) — arXiv:2308.03688
Invoking: `calcular` with `{'expresion': '3 * 2'}`
responded: Los papers más importantes identificados son:

*   **ReAct: Synergizing Reasoning and Acting** (Yao et al., 2022)
*   **AutoGen: Enabling Next-Gen LLM Applications** (Wu et al., 2023)
*   **AgentBench: Evaluating LLMs as Agents** (Liu et al., 2023)

Ahor

**Output esperado:** El agente debería:
1. Llamar a `buscar_papers("multi-agent")` → obtener 3 papers
2. Llamar a `calcular('3 * 2')` → obtener 6
3. Sintetizar ambos resultados en una respuesta coherente, usando el tono de un investigador senior (cortesía del warm-up de la skill)

**Interpretación:** Este proyecto integra los tres conceptos centrales de la notebook en un único sistema:
- **Agente ReAct** — ciclo de razonamiento + tool calls iterativos
- **Skill warm-up** — contexto de investigador senior inyectado via `context_loader`
- **Memoria in-context** — el agente mantiene coherencia entre llamadas a tools gracias al scratchpad

Es un sistema simple (3 tools, 1 skill) pero completamente funcional. En U5_02 añadiremos estado persistente con LangGraph; en U5_05 reemplazaremos `buscar_papers` mock con RAG real sobre arXiv.

---
## 8.5 Smolagents: Agentes Basados en Código <a id='8-5-smolagents'></a>

Hasta aquí usamos LangChain's `AgentExecutor` con tool-calling. HuggingFace lanzó **smolagents** (mar 2025) con un paradigma diferente: el LLM escribe y ejecuta **código Python real** en lugar de llamar a tools discretas.

| Dimensión | LangChain AgentExecutor | smolagents CodeAgent |
|-----------|------------------------|---------------------|
| Acción del LLM | Selecciona nombre de tool + args JSON | Escribe bloque `python` ejecutable |
| Flexibilidad | Limitada a tools registradas | Composición arbitraria de código |
| Trazabilidad | Thought/Action/Observation | Código + stdout como observación |
| Riesgo de seguridad | Bajo (tools fijas) | Mayor (requiere sandbox) |
| Mejor para | Pipelines predecibles | Tareas de análisis ad-hoc |

### 8.5.1 CodeAgent básico con Smolagents


In [25]:
# ============================================================
# Smolagents: CodeAgent — el LLM escribe código, smolagents lo ejecuta
# ============================================================
try:
    from smolagents import CodeAgent, DuckDuckGoSearchTool, OpenAIServerModel
    print("smolagents cargado correctamente")
    _SMOLAGENTS_OK = True
except ImportError:
    print("[Aviso] Instala smolagents: pip install smolagents>=1.13")
    _SMOLAGENTS_OK = False

if _SMOLAGENTS_OK:
    # Paso 1: Definir el modelo con OpenAIServerModel (compatible con OpenRouter)
    import os
    _model_id = os.getenv("OPENROUTER_MODEL", "google/gemini-2.5-flash")
    try:
        model = OpenAIServerModel(
            model_id=_model_id,
            api_key=os.getenv("OPENROUTER_API_KEY", ""),
            api_base="https://openrouter.ai/api/v1",
        )
        print(f"Modelo: {_model_id}")
    except Exception as e:
        # Fallback: modelo vía LiteLLM (Ollama local, no requiere API key)
        from smolagents import LiteLLMModel
        model = LiteLLMModel(model_id="ollama/llama3.2")
        print(f"[Fallback] Usando Ollama llama3.2 (error original: {e})")

    # Paso 2: Crear el CodeAgent con herramienta de búsqueda
    agent_smol = CodeAgent(
        tools=[DuckDuckGoSearchTool()],
        model=model,
        max_steps=4,
        verbosity_level=1,
    )

    # Paso 3: Tarea que combina búsqueda + cálculo — el agente escribe el código
    task = (
        "Busca el radio atómico del oro (Au) en picómetros. "
        "Luego calcula cuántas veces cabe en 1 nanómetro (1000 pm)."
    )
    print(f"Tarea: {task}")
    print("-" * 50)
    try:
        result = agent_smol.run(task)
        print("\nResultado:", result)
    except Exception as e:
        print(f"Error en ejecución: {e}")
        print("(Normal en entornos sin conexión — el flujo CodeAgent es lo importante)")


smolagents cargado correctamente
Modelo: google/gemini-2.5-flash
Tarea: Busca el radio atómico del oro (Au) en picómetros. Luego calcula cuántas veces cabe en 1 nanómetro (1000 pm).
--------------------------------------------------


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Busca el radio atómico del oro (Au) en picómetros. Luego calcula cuántas veces cabe en 1 nanómetro (1000 pm).   │
│                                                                                                                 │
╰─ OpenAIModel - google/gemini-2.5-flash ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  gold_atomic_radius_info = web_search("radio atómico del oro en picómetros")                                      
  print(gold_atomic_radius_info)                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Anexo:Radio atómico de los elementos 
químicos](https://es.wikipedia.org/wiki/Anexo:Radio_atómico_de_los_elementos_químicos)
Elradiode unátomono es una propiedad definida de manera única, por lo que no se pueden comparar datos derivados de 
fuentes con diferentes suposiciones. Todas las medidas están dadas enpicómetros(pm).

[Masa atómica relativa - Wikipedia, la enciclopedia libre](https://es.wikipedia.org/wiki/Masa_atómica_relativa)
La masa atómica relativa (símbolo: Ar), anteriormente conocida como peso atómico, es una magnitud física 
adimensional, definida como la razón del promedio ...

[Tabla de Radios Atómicos e Iónicos | PDF | Minerales de 
...](https://es.scribd.com/document/442459075/TABLA-DE-RADIOS-ATOMICOS-E-IONICOS)
Este documento presenta una tabla con losradiosatómicose iónicos de varios elementos químicos. La tabla incluye el 
elemento, suradioatómicoen angstroms, su valencia más común y elradioiónico en angstroms cuando el elemento forma 
un ión con esa valencia.

[Radios atómicos de los elementos (página de 
datos)](https://academia-lab.com/enciclopedia/radios-atomicos-de-los-elementos-pagina-de-datos/)
A menudo se indica con a0 y son aproximadamente las 53 pm. Por lo tanto, los valores deradiosatómicosdados aquí 
enpicómetrosse pueden convertir a unidades atómicas dividiéndolos por 53, hasta el nivel de precisión de los datos 
proporcionados en esta tabla.

[Elementos químicos ordenados por su radio atómico - 
Lenntech](https://www.lenntech.es/tabla-peiodica/radio-atomico.htm)
Elementos químicos ordenados por suradio atómicoLenntech (European Head Office) Distributieweg 3 2645 EG Delfgauw 
Los Países Bajos Phone: +31 152 755 704 fax: +31 152 616 289 e-mail: info@lenntech.com Lenntech USA LLC (Américas) 
5975 Sunset Drive South Miami, FL 33143 USA Phone: +1 877 453 8095 e-mail: info@lenntech.com Lenntech DMCC (Medio 
Este) Level 6 - OFFICE #101-One JLT Tower ...

[Sólidos cristalinos explicados + Ejercicio: radio atómico ...Oro (Au) - Periodic TableRadio Atómico | Comprensión 
y Medición - modern-physics.org](https://www.youtube.com/watch?v=Wpw2_zYQJyI)
Calcula elradioatómicoenpicómetros.” 🔹 Aprenderás a: Relacionar la longitud de celda con elradioatómico. Calcular 
densidad y volumen de la celda unitaria. Usar correctamente ...Oro(Au) elemento químico con número atómico 79 ... 
Propiedad Física ... Anexo:radioAtómicoDe Los Elementos Químicos:Oro0 10 20 30 40 50 60 70 80 90 100 110 120 130 
140 150 160 170 180 190 200 210 220 pmRadioAtómicoRadio Covalente Metallic Radius Radio de Van der WaalsOct 10, 
2024 ·Elradioatómicoes una medida del tamaño de un átomo, que típicamente se representa enpicómetros(pm) o 
ángstroms (Å). Debido a que los electrones en un átomo no tienen una ubicación fija, definir elradioatómicocon 
precisión puede ser complicado.

[Oro (Au) - Periodic Table](https://periodictable.chemicalaid.com/element.php/Au?lang=es)
Oro(Au) elemento químico con número atómico 79 ... Propiedad Física ... Anexo:radioAtómicoDe Los Elementos 
Químicos:Oro0 10 20 30 40 50 60 70 80 90 100 110 120 130 140 150 160 170 180 190 200 210 220 pmRadioAtómicoRadio 
Covalente Metallic Radius Radio de Van der Waals

[Radio Atómico | Comprensión y Medición - 
modern-physics.org](https://modern-physics.org/radio-atomico-comprension-y-medicion/)
Oct 10, 2024 ·Elradioatómicoes una medida del tamaño de un átomo, que típicamente se representa enpicómetros(pm) o 
ángstroms (Å). Debido a que los electrones en un átomo no tienen una ubicación fija, definir elradioatómicocon 
precisión puede ser complicado.

[Oro (Au) - Periodic Table - ChemicalAid](https://periodictable.chemicalaid.com/element.php/Au?lang=an)
Animated Modelo atomico de Bohr of Au (Oro). Play: Video. Physical Property. Radio Atomico. Rahm. 135 pm · 226 pm. 
molar volume. 10.2 cm³/mol. Radio covalent.

[SOLUTION: Oro wikipedia la enciclopedia libre - 
Studypool](https://www.studypool.com/documents/39273820/oro-wikipedia-la-encic

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  gold_atomic_radius_pm = 135                                                                                      
  nanometer_in_pm = 1000                                                                                           
                                                                                                                   
  times_fits = nanometer_in_pm / gold_atomic_radius_pm                                                             
  final_answer(times_fits)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 7.407407407407407

[Step 2: Duration 1.59 seconds| Input tokens: 6,001 | Output tokens: 248]


Resultado: 7.407407407407407


**¿Por qué importa CodeAgent?**

Cuando el LLM escribe `radio_au = 144  # pm` y luego `print(1000 / radio_au)`,
está razonando **dentro** del lenguaje, no seleccionando tools discretas. Esto permite:
- Reutilizar variables entre pasos sin estado explícito
- Composición arbitraria (loops, filtros, visualizaciones)
- Inspeccionar el razonamiento leyendo el código generado

**Trade-off clave:** Mayor poder expresivo, mayor superficie de ataque. Smolagents
ejecuta el código en un sandbox E2B (en producción) o en el proceso local (en notebooks).
Para producción, usar siempre `E2BExecutor` o contenedor aislado.

**Diferencia con U5_03 CrewAI:** CrewAI coordina *múltiples* agentes con roles fijos;
smolagents enfoca en un *solo* agente que genera código como su mecanismo de acción.


---
## 9. Hiperparámetros de Modelos y Agentes <a id='10-hiperparametros'></a>

Los **hiperparámetros** controlan *cómo* el LLM genera texto — independientemente de lo que diga el prompt.
Ajustarlos correctamente es la diferencia entre una respuesta genérica y una respuesta científicamente precisa.

---

### 9.1 Hiperparámetros de Muestreo (Sampling)

| Hiperparámetro | Rango | Descripción | Uso típico |
|---|---|---|---|
| `temperature` | 0.0 – 2.0 | Controla la aleatoriedad. 0 = determinista, 2 = muy creativo | Ciencia: 0.0–0.2 / Escritura: 0.7–1.2 |
| `top_p` | 0.0 – 1.0 | *Nucleus sampling*: considera solo los tokens cuya prob. acumulada ≤ top_p | 0.1 para respuestas técnicas |
| `top_k` | 1 – ∞ | Considera solo los k tokens más probables en cada paso | 1 = greedy decoding |
| `frequency_penalty` | -2.0 – 2.0 | Penaliza tokens según su frecuencia en el texto generado hasta el momento | Reduce repeticiones |
| `presence_penalty` | -2.0 – 2.0 | Penaliza tokens que ya aparecieron al menos una vez | Fuerza variedad temática |
| `repetition_penalty` | 1.0 – ∞ | Multiplicador que penaliza la repetición (alternativa a frequency/presence) | Modelos HuggingFace |
| `seed` | entero | Semilla para reproducibilidad exacta | Experimentos replicables |
| `n` | 1 – ∞ | Número de respuestas independientes a generar | Voting / ensemble |

---

### 9.2 Hiperparámetros de Longitud y Parada

| Hiperparámetro | Rango | Descripción |
|---|---|---|
| `max_tokens` / `max_completion_tokens` | 1 – límite del modelo | Máximo de tokens en la respuesta |
| `min_tokens` | 0 – max_tokens | Mínimo de tokens (fuerza respuestas más largas) |
| `stop` / `stop_sequences` | lista de strings | El modelo para cuando genera cualquiera de estas cadenas |
| `truncation` | bool | Truncar el prompt si excede la ventana de contexto |

---

### 9.3 Hiperparámetros de Contexto

| Hiperparámetro | Descripción |
|---|---|
| `context_window` | Máximo de tokens de entrada + salida (propiedad del modelo) |
| `stream` | Devolver tokens uno a uno en tiempo real |
| `logit_bias` | Diccionario `{token_id: bias}` para aumentar/suprimir tokens específicos |
| `logprobs` | Devolver log-probabilidades de los tokens generados |
| `echo` | Incluir el prompt en la respuesta (modelos de completion) |

---

### 9.4 Hiperparámetros de Agente (LangChain AgentExecutor)

| Hiperparámetro | Descripción | Valor recomendado |
|---|---|---|
| `max_iterations` | Máximo de pasos razonamiento–acción–observación | 5–10 |
| `max_execution_time` | Tiempo límite en segundos para el agente | 30–120 |
| `early_stopping_method` | `"force"` (corta) o `"generate"` (pide resumen final) | `"generate"` |
| `handle_parsing_errors` | Si falla el parser del LLM, reintenta o usa fallback | `True` |
| `return_intermediate_steps` | Devuelve cadena de pensamiento completa | `True` para debug |
| `verbose` | Imprime cada paso del agente | `True` para desarrollo |
| `trim_intermediate_steps` | Recorta pasos intermedios para ahorrar tokens | -1 (sin límite) |

---

### 9.5 Hiperparámetros de Memoria

| Hiperparámetro | Clase | Descripción |
|---|---|---|
| `k` | `ConversationBufferWindowMemory` | Número de turnos a conservar |
| `max_token_limit` | `ConversationSummaryBufferMemory` | Tokens máximos antes de comprimir |
| `return_messages` | todas | Devolver objetos Message en vez de string plano |
| `human_prefix` / `ai_prefix` | todas | Etiquetas para los turnos en el historial |


In [35]:
# ============================================================
# Comparativa: Hiperparámetros por defecto vs. optimizados para ciencia
# Pregunta: Au 20 nm vs 60 nm — LSPR, S/V y reactividad comparada
# ============================================================
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

PREGUNTA = (
    "Compara nanopartículas de Au de 20 nm y 60 nm: "
    "calcula su LSPR y su relación S/V. "
    "¿Cuál es más reactiva y por qué?"
)

def hacer_pregunta(llm, etiqueta):
    respuesta = llm.invoke(PREGUNTA)
    tokens = getattr(respuesta, 'response_metadata', {}).get('token_usage', {})
    print(f"\n{'='*60}")
    print(f"  {etiqueta}")
    print(f"{'='*60}")
    print(respuesta.content)
    if tokens:
        print(f"\n[Tokens usados: {tokens}]")
    return respuesta

# ── Configuración base (misma para ambos) ──────────────────────────────────
_base_kwargs = dict(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY", ""),
    model=os.environ.get("OPENROUTER_MODEL", "google/gemini-2.5-flash"),
    default_headers={
        "HTTP-Referer": "https://github.com/antigravity-nano",
        "X-Title": "Antigravity Nano IA Course",
    },
)

# ── LLM 1: Parámetros por DEFECTO ─────────────────────────────────────────
llm_default = ChatOpenAI(
    **_base_kwargs,
    temperature=0.7,      # defecto — bastante aleatorio
    # max_tokens no fijado — sin límite explícito
)

# ── LLM 2: Parámetros OPTIMIZADOS para respuestas científicas ─────────────
llm_cientifico = ChatOpenAI(
    **_base_kwargs,
    temperature=0.1,          # casi determinista — reproducible
    max_tokens=512,           # respuesta concisa y enfocada
    top_p=0.1,                # solo tokens de alta probabilidad
    frequency_penalty=0.3,    # evita repetir frases
    presence_penalty=0.1,     # incentiva variedad de conceptos
    model_kwargs={
        "seed": 42,                          # reproducibilidad
        "stop": ["Referencias:", "---"],     # para antes de bibliografía
    },
)

print("Pregunta:", PREGUNTA)

resp_default    = hacer_pregunta(llm_default,    "PARAMETROS POR DEFECTO (temperature=0.7)")
resp_cientifico = hacer_pregunta(llm_cientifico, "PARAMETROS CIENTIFICOS (temperature=0.1, top_p=0.1, max_tokens=512)")


c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\IPython\core\interactiveshell.py:3641: UserWarning: Parameters {'seed', 'stop'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


Pregunta: Compara nanopartículas de Au de 20 nm y 60 nm: calcula su LSPR y su relación S/V. ¿Cuál es más reactiva y por qué?

  PARAMETROS POR DEFECTO (temperature=0.7)
¡Claro! Vamos a comparar nanopartículas de oro (Au) de 20 nm y 60 nm, calculando su LSPR y su relación superficie-volumen (S/V). Luego, discutiremos cuál es más reactiva y por qué.

**1. Resonancia de Plasmones Superficiales Localizados (LSPR)**

El LSPR es la oscilación colectiva de los electrones de conducción en la superficie de una nanopartícula metálica cuando es excitada por luz. La longitud de onda del LSPR depende fuertemente del tamaño, la forma, el material de la nanopartícula y el entorno dieléctrico.

**Cálculo del LSPR:**

Para nanopartículas esféricas de oro, podemos usar una aproximación simplificada del modelo de Mie para estimar el LSPR. Sin embargo, para un cálculo preciso, se requieren métodos numéricos avanzados como FDTD (Finite-Difference Time-Domain).

Una aproximación cualitativa es que **a medid

In [36]:
# ============================================================
# AgentExecutor con hiperparámetros explícitos para investigación
# ============================================================
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool

@tool
def calcular_lspr(diametro_nm: float) -> str:
    """Calcula LSPR aproximada de una nanopartícula de Au (aprox. Mie, 10-80 nm)."""
    diametro_nm = float(diametro_nm)
    if not (10 <= diametro_nm <= 80):
        return f"Advertencia: fuera del rango de validez (10-80 nm). d={diametro_nm} nm"
    lspr_nm = 512 + 1.1 * diametro_nm
    color = 'rojo' if lspr_nm > 590 else 'verde-amarillo'
    return f"Au Ø{diametro_nm} nm → LSPR ≈ {lspr_nm:.1f} nm ({color})"

@tool
def relacion_superficie_volumen(diametro_nm: float) -> str:
    """Calcula S/V de una nanopartícula esférica en nm⁻¹. Alta S/V = más reactividad."""
    diametro_nm = float(diametro_nm)
    sv = 6 / diametro_nm
    return f"Ø{diametro_nm} nm → S/V = {sv:.4f} nm⁻¹"

# ── LLM optimizado para agente científico ─────────────────────────────
llm_agente = ChatOpenAI(
    **_base_kwargs,
    temperature=0.0,    # determinísta: razonamiento lógico reproducible
    max_tokens=1024,    # suficiente para cadena de pensamiento completa
    top_p=0.05,         # muy conservador — solo lo más probable
)

# Prompt ReAct — requiere exactamente: {tools}, {tool_names}, {input}, {agent_scratchpad}
REACT_TEMPLATE = "\n".join([
    "Eres un experto en nanociencia. Responde usando solo datos de las herramientas. Se conciso.",
    "",
    "Herramientas disponibles:",
    "{tools}",
    "",
    "Nombres de herramientas (escribe exactamente este nombre en Action):",
    "{tool_names}",
    "",
    "Formato obligatorio:",
    "Thought: razona el siguiente paso",
    "Action: nombre_herramienta",
    "Action Input: argumento numerico",
    "Observation: resultado",
    "... (repite si necesitas mas pasos)",
    "Thought: tengo todos los datos",
    "Final Answer: respuesta cientifica final",
    "",
    "Pregunta: {input}",
    "{agent_scratchpad}",
])
react_prompt = PromptTemplate.from_template(REACT_TEMPLATE)

agent_react = create_react_agent(
    llm_agente,
    [calcular_lspr, relacion_superficie_volumen],
    react_prompt,
)

# ── AgentExecutor con hiperparámetros de agente ─────────────────────────
executor = AgentExecutor(
    agent=agent_react,
    tools=[calcular_lspr, relacion_superficie_volumen],
    max_iterations=5,                  # máximo 5 pasos razonamiento-acción
    max_execution_time=60,             # timeout 60 segundos
    early_stopping_method="generate",  # genera resumen si se acaban iteraciones
    handle_parsing_errors=True,        # reintenta si el LLM falla el formato
    return_intermediate_steps=True,    # captura cadena de pensamiento completa
    verbose=True,
)

print("Agente científico ejecutándose...")
print("="*60)
resultado = executor.invoke({
    "input": (
        "Compara nanopartículas de Au de 20 nm y 60 nm: "
        "calcula su LSPR y su relación S/V. ¿Cuál es más reactiva y por qué?"
    )
})

print("\n" + "="*60)
print("RESPUESTA FINAL:")
print(resultado["output"])
print(f"\nPasos de razonamiento: {len(resultado['intermediate_steps'])}")

Agente científico ejecutándose...


> Entering new AgentExecutor chain...
Thought: Necesito calcular el LSPR y la relación S/V para nanopartículas de 20 nm y 60 nm. Luego, compararé los resultados para determinar cuál es más reactiva.
Action: calcular_lspr
Action Input: 20Au Ø20.0 nm → LSPR ≈ 534.0 nm (verde-amarillo)Thought: Ya calculé el LSPR para 20 nm. Ahora necesito calcular la relación S/V para 20 nm.
Action: relacion_superficie_volumen
Action Input: 20Ø20.0 nm → S/V = 0.3000 nm⁻¹Thought: Ya calculé el LSPR y la relación S/V para 20 nm. Ahora necesito calcular el LSPR para 60 nm.
Action: calcular_lspr
Action Input: 60Au Ø60.0 nm → LSPR ≈ 578.0 nm (verde-amarillo)Thought: Ya calculé el LSPR para 60 nm. Ahora necesito calcular la relación S/V para 60 nm.
Action: relacion_superficie_volumen
Action Input: 60Ø60.0 nm → S/V = 0.1000 nm⁻¹Thought: Tengo todos los datos necesarios para responder la pregunta.
Final Answer: Para una nanopartícula de Au de 20 nm, el LSPR es aproximadamente 5

In [40]:
# ============================================================
# Tabla resumen: efecto de temperature en la misma pregunta
# ============================================================
temperatures = [0.0, 0.5, 1.0, 1.5]
pregunta_corta = "Define plasmón superficial en una oración."

print(f"Pregunta: {pregunta_corta}")
print("="*70)

for temp in temperatures:
    llm_tmp = ChatOpenAI(
        **_base_kwargs,
        temperature=temp,
        max_tokens=200,
        seed=42,   # misma semilla → diferencia viene solo de temperature
    )
    resp = llm_tmp.invoke(pregunta_corta)
    texto = resp.content.strip().replace("\n", " ")
    print(f"  temperature={temp:.1f} | {texto}")

print("\nObservación: a temperature=0.0 la respuesta es la misma en cada ejecución.")
print("A temperature>=1.0 la variabilidad aumenta notablemente.")


Pregunta: Define plasmón superficial en una oración.
  temperature=0.0 | Un plasmón superficial es una oscilación colectiva de electrones en la interfaz entre un metal y un dieléctrico, que se propaga a lo largo de la superficie.
  temperature=0.5 | Un plasmón superficial es una oscilación colectiva de electrones en la interfaz entre un metal y un dieléctrico, que se propaga a lo largo de dicha interfaz.
  temperature=1.0 | Un plasmón superficial es una oscilación combinada de electrones y ondas electromagnéticas que viajan a lo largo de la interfaz entre un metal y un dieléctrico.
  temperature=1.5 | Un plasmón superficial es una oscilación combinada de electrones y ondas electromagnéticas que viajan a lo largo de la interfaz entre un metal y un dieléctrico.

Observación: a temperature=0.0 la respuesta es la misma en cada ejecución.
A temperature>=1.0 la variabilidad aumenta notablemente.


---
## 10. Resumen y Criterios de Evaluación <a id='9-resumen'></a>

### Conceptos Clave de esta Notebook

| Concepto | Definición en una oración | Implementado en |
|----------|-------------------------|-----------------|
| **Agente** | Sistema que cicla entre percibir, razonar y actuar usando un LLM como política | Sección 3 |
| **ReAct** | Patrón que intercala thought → action → observation en ciclos iterativos | Sección 3 |
| **Tool** | Función que el LLM puede llamar en tiempo de ejecución para actuar en el mundo | Secciones 3, 5, 8 |
| **Skill** | Módulo de warm-up que inyecta expertise de dominio antes de ejecutar | Sección 4 |
| **LCEL** | Sistema de composición de cadenas con el operador `\|` | Sección 5 |
| **4 tipos de memoria** | In-context, episódica, semántica, procedimental | Sección 6 |
| **Chain of Thought** | Instrucción para mostrar pasos de razonamiento antes de responder | Sección 7 |

### Criterios de Evaluación

Habrás completado esta notebook satisfactoriamente cuando puedas:

- [ ] **Construir** un agente ReAct con al menos 2 tools que resuelva una tarea multi-paso
- [ ] **Describir** los 4 tipos de memoria con un ejemplo concreto de cada uno
- [ ] **Registrar** una skill propia (`context_loader` u otra) en el Skill Registry
- [ ] **Demostrar** la diferencia observable entre un agente con y sin skill warm-up
- [ ] **Identificar** los 3 errores comunes de esta sección en código de un compañero

### Ejercicio de Extensión

Crea un agente asistente para un dominio diferente al de los ejemplos:

1. Define 2-3 tools específicas del dominio elegido
2. Añade un nuevo dominio a `DOMAIN_CONTEXTS` en `context_loader.py` con un system prompt de 4+ puntos
3. Crea el agente con tu skill personalizada
4. Muestra la diferencia de respuestas con y sin warm-up para la misma pregunta
5. Registra tu skill en el registry con `SKILL_METADATA` completo

**Ideas de dominio:** derecho (revisión de contratos), finanzas (análisis de inversiones), medicina (síntesis de literatura clínica), educación (tutoría adaptativa), DevOps (análisis de logs).

### Próxima Notebook

En **U5_02 — LangChain Avanzado y LangGraph** construiremos sobre estos fundamentos:
- Workflows con **estado persistente** (el agente recuerda entre ejecuciones, no solo dentro de una)
- **Ciclos condicionales** (el agente puede volver a pasos anteriores si el resultado no es satisfactorio)
- **Human-in-the-Loop** (pausar el workflow para pedir aprobación humana)
- La skill `task_classifier` que enruta automáticamente tasks al agente correcto

---
*Notebook generada siguiendo el PROTOCOLO_UNIDAD_5_MULTIAGENTE.md v1.1.0*  
*Antigravity-Nano-Research — Unit 05 Multi-Agent Systems — Marzo 2026*


---
## 10. Resumen y Criterios de Evaluación <a id='10-resumen'></a>

### Conceptos Clave de esta Notebook

| Concepto | Definición en una oración | Implementado en |
|----------|--------------------------|-----------------|
| **Agente** | Sistema que cicla entre percibir, razonar y actuar usando un LLM como política | Sección 3 |
| **ReAct** | Patrón que intercala thought → action → observation en ciclos iterativos | Sección 3 |
| **Tool** | Función que el LLM puede llamar en tiempo de ejecución para actuar en el mundo | Secciones 3, 5, 8 |
| **Skill** | Módulo de warm-up que inyecta expertise de dominio antes de ejecutar | Sección 4 |
| **LCEL** | Sistema de composición de cadenas con el operador `\|` | Sección 5 |
| **4 tipos de memoria** | In-context, episódica, semántica, procedimental | Sección 6 |
| **Chain of Thought** | Instrucción para mostrar pasos de razonamiento antes de responder | Sección 7 |
| **temperature** | Controla la aleatoriedad del muestreo: 0.0 = determinista, >1.0 = muy creativo | Sección 10 |
| **top_p** | Muestreo de núcleo: restringe a los tokens cuya probabilidad acumulada ≤ p | Sección 10 |
| **frequency_penalty** | Penaliza tokens ya usados con frecuencia — reduce repetición literal | Sección 10 |
| **presence_penalty** | Penaliza tokens que aparecieron al menos una vez — incentiva variedad temática | Sección 10 |
| **max_tokens** | Límite duro de tokens en la respuesta — controla longitud y costo | Sección 10 |
| **max_iterations** | Máximo de ciclos razonamiento-acción en AgentExecutor | Sección 10 |
| **seed** | Fija la semilla del generador para reproducibilidad entre ejecuciones | Sección 10 |

### Criterios de Evaluación

Habrás completado esta notebook satisfactoriamente cuando puedas:

- [ ] **Construir** un agente ReAct con al menos 2 tools que resuelva una tarea multi-paso
- [ ] **Describir** los 4 tipos de memoria con un ejemplo concreto de cada uno
- [ ] **Registrar** una skill propia (`context_loader` u otra) en el Skill Registry
- [ ] **Demostrar** la diferencia observable entre un agente con y sin skill warm-up
- [ ] **Identificar** los 3 errores comunes de esta sección en código de un compañero
- [ ] **Explicar** qué hiperparámetro ajustarías para hacer una respuesta más corta, más reproducible, o menos repetitiva — y por qué
- [ ] **Comparar** el output de `llm_default` vs. `llm_cientifico` para la misma pregunta y señalar las diferencias concretas
- [ ] **Demostrar** que un agente con herramientas produce valores exactos donde un LLM sin herramientas solo estima

### Ejercicio de Extensión

Crea un agente asistente para un dominio diferente al de los ejemplos:

1. Define 2-3 tools específicas del dominio elegido
2. Añade un nuevo dominio a `DOMAIN_CONTEXTS` en `context_loader.py` con un system prompt de 4+ puntos
3. Crea el agente con tu skill personalizada
4. Muestra la diferencia de respuestas con y sin warm-up para la misma pregunta
5. Registra tu skill en el registry con `SKILL_METADATA` completo
6. Ajusta los hiperparámetros (`temperature`, `top_p`, `frequency_penalty`, `max_tokens`) a los valores óptimos para tu dominio y justifica cada elección

**Ideas de dominio:** derecho (revisión de contratos), finanzas (análisis de inversiones), medicina (síntesis de literatura clínica), educación (tutoría adaptativa), DevOps (análisis de logs).

### Próxima Notebook

En **U5_02 — LangChain Avanzado y LangGraph** construiremos sobre estos fundamentos:
- Workflows con **estado persistente** (el agente recuerda entre ejecuciones, no solo dentro de una)
- **Ciclos condicionales** (el agente puede volver a pasos anteriores si el resultado no es satisfactorio)
- **Human-in-the-Loop** (pausar el workflow para pedir aprobación humana)
- La skill `task_classifier` que enruta automáticamente tasks al agente correcto

---
*Notebook generada siguiendo el PROTOCOLO_UNIDAD_5_MULTIAGENTE.md v1.1.0*  
*Antigravity-Nano-Research — Unit 05 Multi-Agent Systems — Marzo 2026*
